In [0]:
ruta_empleados = "/Volumes/electrocasa_dev/bronze/landing/empleados/empleados_rrhh.csv"

empleados = (
    spark.read
        .option("header", "true")
        .csv(ruta_empleados)
)

empleados.printSchema()
display(empleados.limit(10))

In [0]:
empleados.createOrReplaceTempView("empleados_tmp")

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT id_empleado) AS empleados_unicos,

    SUM(CASE
        WHEN dni IS NULL OR TRIM(dni) = ''
        THEN 1 ELSE 0
    END) AS dni_faltante,

    SUM(CASE
        WHEN fecha_evento IS NULL OR TRIM(fecha_evento) = ''
        THEN 1 ELSE 0
    END) AS fecha_evento_faltante,

    SUM(CASE
        WHEN salario IS NULL OR TRIM(salario) = ''
        THEN 1 ELSE 0
    END) AS salario_faltante

FROM empleados_tmp;

In [0]:
%sql

SELECT
    tipo_evento,
    COUNT(*) AS cantidad
FROM empleados_tmp
GROUP BY tipo_evento
ORDER BY tipo_evento;

In [0]:
%sql

SELECT
    cantidad_eventos,
    COUNT(*) AS cantidad_empleados
FROM (
    SELECT
        id_empleado,
        COUNT(*) AS cantidad_eventos
    FROM empleados_tmp
    GROUP BY id_empleado
)
GROUP BY cantidad_eventos
ORDER BY cantidad_eventos;

In [0]:
%sql

SELECT
    COUNT(*) AS pares_empleado_fecha_repetidos
FROM (
    SELECT
        id_empleado,
        fecha_evento,
        COUNT(*) AS cantidad
    FROM empleados_tmp
    WHERE fecha_evento IS NOT NULL
      AND TRIM(fecha_evento) <> ''
    GROUP BY id_empleado, fecha_evento
    HAVING COUNT(*) > 1
);

In [0]:
%sql

SELECT e.*
FROM empleados_tmp e
INNER JOIN (
    SELECT
        id_empleado,
        fecha_evento
    FROM empleados_tmp
    WHERE fecha_evento IS NOT NULL
      AND TRIM(fecha_evento) <> ''
    GROUP BY id_empleado, fecha_evento
    HAVING COUNT(*) > 1
) d
ON e.id_empleado = d.id_empleado
AND e.fecha_evento = d.fecha_evento
ORDER BY e.id_empleado, e.fecha_evento, e.tipo_evento;

In [0]:
%sql

SELECT
    id_empleado,
    fecha_evento,
    lower(
        regexp_replace(
            trim(tipo_evento),
            ' ',
            '_'
        )
    ) AS tipo_evento_normalizado,
    COUNT(*) AS cantidad
FROM empleados_tmp
WHERE fecha_evento IS NOT NULL
  AND TRIM(fecha_evento) <> ''
GROUP BY
    id_empleado,
    fecha_evento,
    lower(
        regexp_replace(
            trim(tipo_evento),
            ' ',
            '_'
        )
    )
HAVING COUNT(*) > 1
ORDER BY id_empleado;